# LEET-Arg plain-model baseline (Colab)

Thin wrapper around `run_leet_arg.py`. All logic lives in the `leet_arg/` package,
so this notebook only clones, installs, authenticates and shells out. Moving this
run to Chameleon Cloud should be a config change, not a rewrite.

This runs the **plain-model, local-SLM** cell of the 2x2 (no solver, no constrained
decoding, no frontier API).

**Runtime:** set `Runtime > Change runtime type > T4 GPU`. Llama-3.2-1B fits
comfortably in any Colab GPU tier.

In [ ]:
#@title 1. Check the GPU
!nvidia-smi || echo 'No GPU detected -- set Runtime > Change runtime type > T4 GPU'

In [ ]:
#@title 2. Clone the branch
BRANCH = 'leet-arg-harness'  #@param {type:"string"}
REPO = 'https://github.com/xai-privacy/analysis-framework.git'  #@param {type:"string"}

import os

if not os.path.isdir('/content/analysis-framework'):
    !git clone --branch {BRANCH} {REPO} /content/analysis-framework
else:
    !cd /content/analysis-framework && git fetch origin && git checkout {BRANCH} && git pull

os.chdir('/content/analysis-framework')
!git log --oneline -1

In [ ]:
#@title 3. Install dependencies
# Colab already ships torch; only transformers/accelerate are usually needed.
!pip install -q transformers accelerate

In [ ]:
#@title 4. Authenticate to Hugging Face
# Llama-3.2 is a gated model: request access on the model page first, then paste
# a token with 'read' scope. In Colab you can instead store it as the secret
# HF_TOKEN and enable notebook access.
from huggingface_hub import notebook_login

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('Using HF_TOKEN from Colab secrets.')
except Exception:
    notebook_login()

In [ ]:
#@title 5. Verify gold label derivation (no model needed)
# Reports how many of the 97 records derive cleanly and names any that do not.
!python3 run_leet_arg.py --verify-gold-only

In [ ]:
#@title 6. Run the baseline
MODEL = 'meta-llama/Llama-3.2-1B-Instruct'  #@param {type:"string"}
N_RECORDS = 10  #@param {type:"integer"}
TRIALS = 5  #@param {type:"integer"}
TEMPERATURE = 0.7  #@param {type:"number"}

!python3 run_leet_arg.py \
    --model {MODEL} \
    --n {N_RECORDS} \
    --trials {TRIALS} \
    --temperature {TEMPERATURE}

In [ ]:
#@title 7. Inspect the newest results file
import glob, json

paths = sorted(glob.glob('results/*.jsonl'))
if not paths:
    print('No results yet -- run the previous cell.')
else:
    latest = paths[-1]
    rows = [json.loads(line) for line in open(latest, encoding='utf-8')]
    print(f'{latest}: {len(rows)} rows\n')
    for row in rows[:3]:
        print(json.dumps(row, indent=2, ensure_ascii=False)[:600])
        print('-' * 60)

In [ ]:
#@title 8. Download the results
try:
    from google.colab import files
    for path in sorted(glob.glob('results/*')):
        files.download(path)
except Exception as exc:
    print(f'Not running in Colab or download unavailable: {exc}')